# GWLR: Risiko *Shortfall* Hasil Padi dan Anomali Iklim di Indonesia

Notebook ini mereproduksi **semua perhitungan** dalam naskah
*"Spatially varying rice yield shortfall risk and climate anomalies in Indonesia: Implications for agricultural insurance premiums"*.

**Cara pakai**
1. Jalankan sel-sel secara berurutan (*Runtime → Run all*).
2. Saat diminta, unggah dua file: `master_agriculture_dataset_v2.xlsx` dan `acri_dataset.xlsx`.
3. Di akhir, file hasil (tabel Excel, gambar PNG, dataset bersih) akan diunduh otomatis.

**Isi**
1. Persiapan dan unggah data
2. Pembersihan data produksi (BPS)
3. Penggabungan dengan ACRI dan konsolidasi Papua
4. Koreksi data 2020
5. Koordinat centroid provinsi
6. Variabel respons: deviasi dari tren dan *shortfall*
7. Fungsi model: logit global, GWLR, GWR, uji Monte Carlo
8. Analisis utama dan uji ketahanan ambang (Tabel 2, 3, 5)
9. Model kontinu: OLS vs GWR (Tabel 4)
10. Statistik deskriptif (Tabel 1) dan ekspor tabel
11. Gambar 1–3
12. Validasi silang dengan `statsmodels`
13. Unduh hasil

> Semua angka acak (uji Monte Carlo) memakai *seed* 2026 dengan urutan yang sama seperti analisis naskah, sehingga hasilnya identik.

## 1. Persiapan dan unggah data

In [ ]:
import numpy as np, pandas as pd, json, os, warnings
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
warnings.filterwarnings('ignore')

OUT = 'hasil'                      # folder output
os.makedirs(OUT, exist_ok=True)

# --- Unggah file (Colab). Di luar Colab, isi path secara manual. ---
PROD_FILE = os.environ.get('PROD_FILE')
ACRI_FILE = os.environ.get('ACRI_FILE')
if PROD_FILE is None:
    try:
        from google.colab import files
        print('Unggah master_agriculture_dataset_v2.xlsx dan acri_dataset.xlsx')
        up = files.upload()
        for name in up:
            if 'acri' in name.lower(): ACRI_FILE = name
            else: PROD_FILE = name
    except ImportError:
        PROD_FILE, ACRI_FILE = 'master_agriculture_dataset_v2.xlsx', 'acri_dataset.xlsx'
print('Produksi :', PROD_FILE); print('ACRI     :', ACRI_FILE)

## 2. Pembersihan data produksi (BPS)
- Menghapus 2 baris catatan kaki BPS yang ikut tersalin ("Keterangan…", "Kualitas produksi…").
- Mengubah tanda "–" menjadi nilai kosong (NaN).

In [ ]:
p = pd.read_excel(PROD_FILE)
p = p[~p.Province.astype(str).str.startswith(('Keterangan', 'Kualitas'))].copy()
for c in ['HarvestArea_ha', 'Productivity_qu_ha', 'Production_ton']:
    p[c] = pd.to_numeric(p[c], errors='coerce')
print(p.shape, '| provinsi unik:', p.Province.nunique())

## 3. Penggabungan dengan ACRI dan konsolidasi Papua
Data ACRI memakai batas Papua sebelum pemekaran 2022, jadi enam provinsi Papua digabung menjadi
**Papua (konsolidasi)** dan **Papua Barat (konsolidasi)**. Luas panen dan produksi dijumlahkan,
lalu produktivitas dihitung ulang = produksi / luas panen × 10 (kuintal/ha).

In [ ]:
a = pd.read_excel(ACRI_FILE)
papua = {'Papua': 'Papua (konsolidasi)', 'Papua Selatan': 'Papua (konsolidasi)',
         'Papua Tengah': 'Papua (konsolidasi)', 'Papua Pegunungan': 'Papua (konsolidasi)',
         'Papua Barat': 'Papua Barat (konsolidasi)', 'Papua Barat Daya': 'Papua Barat (konsolidasi)'}

# cek inkonsistensi produktivitas (terutama tahun 2020)
bad = (p.Productivity_qu_ha - p.Production_ton / p.HarvestArea_ha * 10).abs() > 1
print('Baris dengan produktivitas tidak konsisten:', bad.sum(), p[bad].Year.value_counts().to_dict())

p['Provinsi'] = p.Province.replace(papua)
g = (p.groupby(['Provinsi', 'Year'], as_index=False)
       .agg(Region=('Region', 'first'), HarvestArea_ha=('HarvestArea_ha', 'sum'),
            Production_ton=('Production_ton', 'sum')))
g['Productivity_qu_ha'] = g.Production_ton / g.HarvestArea_ha * 10
m = a.merge(g.rename(columns={'Year': 'Tahun'}), on=['Provinsi', 'Tahun'], how='left')
print('Hasil gabungan:', m.shape, '| produktivitas kosong:', m.Productivity_qu_ha.isna().sum())

## 4. Koreksi data 2020
Luas panen 2020 di DKI Jakarta, Kalimantan Utara, dan Kepulauan Riau tidak wajar
(contoh: Kalimantan Utara 412 ha, tahun lain ±9.000–13.000 ha). Produktivitas ketiganya
diimputasi dengan interpolasi linier 2019–2021. **Cek ulang angka ini ke publikasi BPS.**

In [ ]:
m['Flag_QC'] = ''
m.loc[m.Tahun == 2020, 'Flag_QC'] = 'Produktivitas 2020 dihitung ulang (Produksi/Luas)'
badarea = (m.Tahun == 2020) & m.Provinsi.isin(['DKI Jakarta', 'Kalimantan Utara', 'Kepulauan Riau'])
m.loc[badarea, 'Flag_QC'] = 'Luas panen 2020 tidak wajar; produktivitas diimputasi (interpolasi 2019-2021)'
m.loc[badarea, 'Productivity_qu_ha'] = np.nan
m = m.sort_values(['Provinsi', 'Tahun']).reset_index(drop=True)
m['Productivity_qu_ha'] = m.groupby('Provinsi').Productivity_qu_ha.transform(lambda s: s.interpolate())
m.loc[badarea, ['Provinsi', 'Tahun', 'Productivity_qu_ha']]

## 5. Koordinat centroid provinsi
Centroid perkiraan (bujur, lintang). Untuk naskah final, **ganti dengan centroid dari shapefile BIG**
lalu jalankan ulang seluruh notebook.

In [ ]:
C = {'Aceh':(96.91,4.23),'Sumatera Utara':(99.06,2.19),'Sumatera Barat':(100.46,-0.86),'Riau':(101.80,0.50),
 'Jambi':(102.72,-1.70),'Sumatera Selatan':(104.17,-3.21),'Bengkulu':(102.36,-3.57),'Lampung':(105.02,-4.91),
 'Kepulauan Bangka Belitung':(106.53,-2.44),'Kepulauan Riau':(104.50,0.90),'DKI Jakarta':(106.84,-6.21),
 'Jawa Barat':(107.60,-6.92),'Jawa Tengah':(110.21,-7.26),'DI Yogyakarta':(110.45,-7.89),'Jawa Timur':(112.72,-7.73),
 'Banten':(106.12,-6.45),'Bali':(115.13,-8.36),'Nusa Tenggara Barat':(117.52,-8.61),'Nusa Tenggara Timur':(122.19,-9.26),
 'Kalimantan Barat':(111.12,-0.09),'Kalimantan Tengah':(113.42,-1.60),'Kalimantan Selatan':(115.44,-2.99),
 'Kalimantan Timur':(116.40,0.50),'Kalimantan Utara':(116.50,3.00),'Sulawesi Utara':(124.49,1.22),
 'Sulawesi Tengah':(121.18,-0.98),'Sulawesi Selatan':(120.00,-3.70),'Sulawesi Tenggara':(122.07,-4.11),
 'Gorontalo':(122.38,0.69),'Sulawesi Barat':(119.30,-2.50),'Maluku':(129.78,-4.69),'Maluku Utara':(127.54,0.22),
 'Papua Barat (konsolidasi)':(133.10,-2.15),'Papua (konsolidasi)':(138.00,-4.20)}

m['Longitude'] = m.Provinsi.map(lambda q: C[q][0])
m['Latitude']  = m.Provinsi.map(lambda q: C[q][1])
assert m.Longitude.notna().all()

provs = sorted(m.Provinsi.unique()); L = len(provs)
loc = m.Provinsi.map({q: i for i, q in enumerate(provs)}).values     # indeks lokasi tiap observasi
xy = np.array([C[q] for q in provs])

def haversine(A, B):
    lon1, lat1, lon2, lat2 = map(np.radians, [A[:, None, 0], A[:, None, 1], B[None, :, 0], B[None, :, 1]])
    h = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(h))
D = haversine(xy, xy)          # matriks jarak antarprovinsi (km), 34 x 34
print(L, 'provinsi; jarak maks', round(D.max()), 'km')

## 6. Variabel respons
- **Yield_dev_trend** = (Y − Ŷ)/Ŷ, Ŷ = tren linier per provinsi (Persamaan 1).
- **Shortfall_5pct** = 1 jika deviasi < −5%.
- Anomali berarah: **Dry** = −anomali curah hujan, **Heat** = anomali suhu.

In [ ]:
def detrend(s):
    t = np.arange(len(s)); f = np.polyval(np.polyfit(t, s.values, 1), t)
    return pd.Series((s.values - f) / f, index=s.index)

m['Yield_dev_trend'] = m.groupby('Provinsi').Productivity_qu_ha.transform(detrend)
m['Shortfall_5pct'] = (m.Yield_dev_trend < -0.05).astype(int)
m['Dry'] = -m.Anomali_Rainfall
m['Heat'] = m.Anomali_Temperature
n = len(m)
print('n =', n, '| shortfall =', m.Shortfall_5pct.sum(), f'({m.Shortfall_5pct.mean():.1%})',
      '| provinsi tanpa shortfall =', (m.groupby('Provinsi').Shortfall_5pct.sum() == 0).sum())
m.to_excel(f'{OUT}/dataset_gabungan_bersih.xlsx', index=False)

## 7. Fungsi model
- `glogit` : logit global (maksimum likelihood via IRLS) — Persamaan 3.
- `gwlr`   : *geographically weighted logistic regression* — Persamaan 4–5. Kernel bisquare adaptif;
  bandwidth = jarak ke provinsi terdekat ke-k. Delapan observasi tahunan satu provinsi berbagi bobot yang sama.
- AICc (Persamaan 6) dengan K = trace matriks *hat* (pendekatan Nakaya et al., 2005).
- `gwr`    : GWR Gaussian untuk hasil kontinu.
- `mc_test`: uji Monte Carlo non-stasioneritas (Fotheringham et al., 1998): acak ulang lokasi provinsi,
  hitung ulang SD koefisien lokal.

In [ ]:
def zs(v): return (v - v.mean()) / v.std()

def irls(X, y, w, ridge=1e-6, it=100):
    b = np.zeros(X.shape[1])
    for _ in range(it):
        eta = X @ b; pr = 1/(1 + np.exp(-eta)); aa = np.clip(pr*(1-pr), 1e-10, None)
        z = eta + (y - pr)/aa
        XtWA = X.T * (w*aa)
        bn = np.linalg.solve(XtWA @ X + ridge*np.eye(len(b)), XtWA @ z)
        if np.max(abs(bn - b)) < 1e-8: b = bn; break
        b = bn
    return b

def deviance(y, pr):
    pr = np.clip(pr, 1e-12, 1-1e-12); return -2*np.sum(y*np.log(pr) + (1-y)*np.log(1-pr))

def glogit(X, y):
    b = irls(X, y, np.ones(len(y)), ridge=0)
    pr = 1/(1 + np.exp(-X @ b)); aa = pr*(1-pr)
    se = np.sqrt(np.diag(np.linalg.inv((X.T*aa) @ X))); k = X.shape[1]; d = deviance(y, pr); nn = len(y)
    return dict(b=b, se=se, p=2*stats.norm.sf(abs(b/se)), dev=d, aic=d+2*k,
                aicc=d + 2*k + 2*k*(k+1)/(nn-k-1), prob=pr)

def kernel(d, k):
    h = np.sort(d)[k-1]*1.0001 if k < L else d.max()*1.0001
    return np.where(d < h, (1 - (d/h)**2)**2, 0.0)

def gwlr(X, y, k, Dm=D):
    nn, q = X.shape; B = np.zeros((L, q)); SE = np.zeros((L, q)); prob = np.zeros(nn); trS = 0.0
    for i in range(L):
        w = kernel(Dm[i], k)[loc]
        b = irls(X, y, w); B[i] = b
        pa = 1/(1 + np.exp(-X @ b)); aa = np.clip(pa*(1-pa), 1e-10, None)
        M = np.linalg.inv((X.T*(w*aa)) @ X + 1e-6*np.eye(q))
        SE[i] = np.sqrt(np.diag(M @ ((X.T*(w*w*aa)) @ X) @ M))
        own = loc == i; prob[own] = pa[own]
        trS += np.sum(np.einsum('ij,jk,ik->i', X[own], M, X[own]) * aa[own])
    d = deviance(y, prob)
    return dict(B=B, SE=SE, prob=prob, dev=d, trS=trS, aic=d+2*trS,
                aicc=d + 2*trS + 2*trS*(trS+1)/(nn-trS-1), k=k)

def ols(X, y):
    b = np.linalg.lstsq(X, y, rcond=None)[0]; e = y - X @ b; k = X.shape[1]; nn = len(y)
    se = np.sqrt(np.diag(e@e/(nn-k) * np.linalg.inv(X.T @ X)))
    aicc = nn*np.log(e@e/nn) + nn*np.log(2*np.pi) + nn*(nn+k)/(nn-2-k)
    return b, se, 2*stats.t.sf(abs(b/se), nn-k), aicc, 1 - e@e/((y-y.mean()) @ (y-y.mean()))

def gwr(X, y, k, Dm=D):
    nn, q = X.shape; B = np.zeros((L, q)); SE = np.zeros((L, q)); yhat = np.zeros(nn); trS = 0; CC = []
    for i in range(L):
        w = kernel(Dm[i], k)[loc]
        Cm = np.linalg.inv((X.T*w) @ X) @ (X.T*w); B[i] = Cm @ y; CC.append(Cm)
        own = loc == i; yhat[own] = X[own] @ B[i]; trS += np.einsum('ij,ji->', X[own], Cm[:, own])
    e = y - yhat; s2 = e@e/(nn-trS)
    for i in range(L): SE[i] = np.sqrt(np.diag(CC[i] @ CC[i].T) * s2)
    aicc = nn*np.log(e@e/nn) + nn*np.log(2*np.pi) + nn*(nn+trS)/(nn-2-trS)
    return dict(B=B, SE=SE, aicc=aicc, trS=trS, r2=1 - e@e/((y-y.mean()) @ (y-y.mean())), k=k)

def best_bw(fitfun, X, y):
    fits = [fitfun(X, y, k) for k in range(8, L+1)]
    return min(fits, key=lambda f: f['aicc']), fits

rng = np.random.default_rng(2026)   # seed tunggal; urutan pemanggilan sama dengan analisis naskah
def mc_test(fitfun, X, y, k, B0, reps=199):
    sd = B0.std(0); c = np.zeros(X.shape[1])
    for _ in range(reps):
        pm = rng.permutation(L)
        c += fitfun(X, y, k, Dm=D[np.ix_(pm, pm)])['B'].std(0) >= sd
    return (c + 1)/(reps + 1)

def auc(y, pr):
    r = stats.rankdata(pr); n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1*(n1+1)/2) / (n1*n0)

## 8. Analisis utama (ambang 5%) dan uji ketahanan ambang 3%, 7%, 10%
Satu loop menghitung logit global, GWLR dengan bandwidth optimal (AICc), dan uji Monte Carlo
(199 permutasi) untuk setiap ambang. Hasil ambang 5% adalah analisis utama (Tabel 2, 3; Gambar 2, 3).
Waktu jalan ±2–4 menit.

In [ ]:
rob = []
for thr in [0.03, 0.05, 0.07, 0.10]:
    y = (m.Yield_dev_trend < -thr).astype(float).values
    X = np.column_stack([np.ones(n), zs(m.ACRI.values)])
    g = glogit(X, y); b, fits = best_bw(gwlr, X, y)
    mcp = mc_test(gwlr, X, y, b['k'], b['B'])
    rob.append(dict(threshold=f'{int(thr*100)}%', events=int(y.sum()),
                    glob_b=g['b'][1], glob_se=g['se'][1], glob_p=g['p'][1], glob_AICc=g['aicc'],
                    gwlr_k=b['k'], gwlr_trS=b['trS'], gwlr_AICc=b['aicc'], dAICc=g['aicc']-b['aicc'],
                    acri_local_min=b['B'][:, 1].min(), acri_local_max=b['B'][:, 1].max(),
                    acri_local_sig=int((abs(b['B'][:, 1]/b['SE'][:, 1]) > 1.96).sum()),
                    mc_p_int=mcp[0], mc_p_acri=mcp[1]))
    if thr == 0.05:
        main = dict(g=g, b=b, fits=fits, y=y, X=X, mcp=mcp)
    print(f'ambang {thr:.0%} selesai')
rob = pd.DataFrame(rob)

g, b, y = main['g'], main['b'], main['y']
bi, _ = best_bw(gwlr, np.ones((n, 1)), y)          # GWLR intercept saja (pembanding)
print(f"\nGLOBAL : b_ACRI={g['b'][1]:.3f} (SE {g['se'][1]:.3f}, p={g['p'][1]:.3f})  AICc={g['aicc']:.2f}  AUC={auc(y, g['prob']):.3f}")
print(f"GWLR   : k={b['k']}  K={b['trS']:.2f}  AICc={b['aicc']:.2f}  AUC={auc(y, b['prob']):.3f}")
print(f"GWLR intercept saja : k={bi['k']}  AICc={bi['aicc']:.2f}")
print(f"Monte Carlo p: intercept={main['mcp'][0]:.3f}, ACRI={main['mcp'][1]:.3f}")
rob.round(3)

## 9. Model kontinu: OLS vs GWR (Tabel 4)

In [ ]:
yc = m.Yield_dev_trend.values * 100; cont = []
for lab, cols in [('ACRI', ['ACRI']), ('Dryness + Heat', ['Dry', 'Heat'])]:
    X = np.column_stack([np.ones(n)] + [zs(m[c].values) for c in cols])
    bo, se, pv, aicc, r2 = ols(X, yc)
    bg, _ = best_bw(gwr, X, yc)
    pm = mc_test(gwr, X, yc, bg['k'], bg['B'])
    for j, c in enumerate(['Intercept'] + cols):
        cont.append(dict(model=lab, term=c, ols_b=bo[j], ols_se=se[j], ols_p=pv[j], ols_AICc=aicc, ols_R2=r2,
                         gwr_k=bg['k'], gwr_AICc=bg['aicc'], gwr_R2=bg['r2'],
                         local_min=bg['B'][:, j].min(), local_max=bg['B'][:, j].max(), mc_p=pm[j]))
cont = pd.DataFrame(cont); cont.round(3)

## 10. Statistik deskriptif (Tabel 1), estimasi lokal, dan ekspor semua tabel

In [ ]:
desc = m[['Productivity_qu_ha', 'Yield_dev_trend', 'Shortfall_5pct', 'ACRI', 'Dry', 'Heat',
          'Rainfall_mm', 'Temperature_C', 'Humidity_pct']].describe().T

t2 = pd.DataFrame({
 'Statistic': ['ACRI coefficient (per SD)', 'Std. error', 'p-value', 'Deviance', 'Effective parameters',
               'AIC', 'AICc', 'AUC', 'Bandwidth (nearest provinces)'],
 'Global logit': [g['b'][1], g['se'][1], g['p'][1], g['dev'], 2, g['aic'], g['aicc'], auc(y, g['prob']), np.nan],
 'GWLR': [f"{b['B'][:,1].min():.3f} to {b['B'][:,1].max():.3f} (median {np.median(b['B'][:,1]):.3f})",
          np.nan, np.nan, b['dev'], b['trS'], b['aic'], b['aicc'], auc(y, b['prob']), b['k']]})

loc_t = pd.DataFrame({'Provinsi': provs, 'Longitude': xy[:, 0], 'Latitude': xy[:, 1],
  'Local_intercept': b['B'][:, 0], 't_intercept': b['B'][:, 0]/b['SE'][:, 0],
  'Local_ACRI': b['B'][:, 1], 't_ACRI': b['B'][:, 1]/b['SE'][:, 1],
  'Baseline_prob_at_mean_ACRI': 1/(1 + np.exp(-b['B'][:, 0])),
  'Observed_shortfall_rate': [y[loc == i].mean() for i in range(L)],
  'Mean_ACRI': [m.ACRI.values[loc == i].mean() for i in range(L)]})

mcdf = pd.DataFrame({'Term': ['Intercept', 'ACRI'], 'SD of local estimates': b['B'].std(0),
                     'Monte Carlo p (199 perms)': main['mcp']})

with pd.ExcelWriter(f'{OUT}/tabel_hasil.xlsx') as xw:
    desc.round(3).to_excel(xw, sheet_name='T1_Deskriptif')
    t2.to_excel(xw, sheet_name='T2_Global_vs_GWLR', index=False)
    mcdf.to_excel(xw, sheet_name='T3_MonteCarlo', index=False)
    cont.to_excel(xw, sheet_name='T4_GWR_kontinu', index=False)
    rob.to_excel(xw, sheet_name='T5_Robustness', index=False)
    loc_t.sort_values('Local_ACRI').to_excel(xw, sheet_name='Lampiran_Lokal', index=False)
print('Tersimpan:', f'{OUT}/tabel_hasil.xlsx')
loc_t.sort_values('Baseline_prob_at_mean_ACRI', ascending=False).round(3).head(10)

## 11. Gambar 1–3
Peta latar diambil dari GeoJSON provinsi (BAKOSURTANAL, via GitHub). Jika gagal diunduh,
gambar tetap dibuat tanpa peta latar. Untuk naskah final, sebaiknya buat peta choropleth dari shapefile BIG.

In [ ]:
import urllib.request
rings = []
try:
    url = 'https://raw.githubusercontent.com/superpikar/indonesia-geojson/master/indonesia-province-simple.json'
    gj = json.load(urllib.request.urlopen(url, timeout=20))
    for f in gj['features']:
        geom = f['geometry']; polys = [geom['coordinates']] if geom['type'] == 'Polygon' else geom['coordinates']
        rings += [np.array(pg[0]) for pg in polys]
    print('Peta latar:', len(rings), 'poligon')
except Exception as e:
    print('Peta latar tidak tersedia:', e)

seq = LinearSegmentedColormap.from_list('seq', ['#cde2fb', '#86b6ef', '#3987e5', '#1c5cab', '#0d366b'])
div = LinearSegmentedColormap.from_list('div', ['#1c5cab', '#86b6ef', '#f0efec', '#f0a3a2', '#b8292a'])
plt.rcParams.update({'font.size': 9, 'axes.edgecolor': '#bdbdbd'})

def base(ax):
    for r in rings: ax.fill(r[:, 0], r[:, 1], color='#ecebe8', ec='#c9c8c4', lw=0.4, zorder=1)
    ax.set_xlim(94.5, 141.5); ax.set_ylim(-11.5, 6.5); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)

def pts(ax, v, cmap, norm=None, sig=None):
    ec = '#fcfcfb' if sig is None else np.where(sig, '#1a1a19', '#fcfcfb')
    return ax.scatter(xy[:, 0], xy[:, 1], c=v, cmap=cmap, norm=norm, s=70, zorder=3, linewidths=1.2, edgecolors=ec)

def cbar(fig, sc, ax, lab):
    cb = fig.colorbar(sc, ax=ax, orientation='horizontal', fraction=0.045, pad=0.02, aspect=40)
    cb.set_label(lab, fontsize=8); cb.outline.set_visible(False)

# Gambar 1
fig, axs = plt.subplots(2, 1, figsize=(7.2, 6.4))
base(axs[0]); cbar(fig, pts(axs[0], loc_t.Observed_shortfall_rate.values, seq), axs[0], 'Share of years with yield >5% below trend')
axs[0].set_title('(a) Observed frequency of rice yield shortfall, 2018–2025', loc='left')
base(axs[1]); cbar(fig, pts(axs[1], loc_t.Mean_ACRI.values, seq), axs[1], 'Mean ACRI (0–1)')
axs[1].set_title('(b) Mean Agricultural Climate Risk Index (ACRI), 2018–2025', loc='left')
fig.tight_layout(); fig.savefig(f'{OUT}/Fig1_observed.png', dpi=300); plt.show()

# Gambar 2
fig, axs = plt.subplots(2, 1, figsize=(7.2, 6.4))
base(axs[0]); cbar(fig, pts(axs[0], loc_t.Baseline_prob_at_mean_ACRI.values, seq), axs[0], 'Probability')
axs[0].set_title('(a) GWLR local baseline probability of shortfall (at mean ACRI)', loc='left')
lim = np.abs(loc_t.Local_ACRI).max()
base(axs[1]); cbar(fig, pts(axs[1], loc_t.Local_ACRI.values, div, TwoSlopeNorm(0, -lim, lim), np.abs(loc_t.t_ACRI.values) > 1.96),
                   axs[1], 'Coefficient (blue = lower risk, red = higher risk)')
axs[1].set_title('(b) GWLR local ACRI coefficient (log-odds per SD); outlined = significant at 5%', loc='left')
fig.tight_layout(); fig.savefig(f'{OUT}/Fig2_gwlr_local.png', dpi=300); plt.show()

# Gambar 3
ks = [f['k'] for f in main['fits']]; ac = [f['aicc'] for f in main['fits']]
fig, ax = plt.subplots(figsize=(6.0, 3.2))
ax.plot(ks, ac, color='#2a78d6', lw=2, marker='o', ms=4)
ax.axhline(g['aicc'], color='#8a8a85', lw=1.2, ls='--'); ax.text(33.6, g['aicc']+0.35, 'Global logit', ha='right', fontsize=8)
ax.axhline(bi['aicc'], color='#8a8a85', lw=1.2, ls=':'); ax.text(33.6, bi['aicc']+0.35, 'GWLR, intercept only (best)', ha='right', fontsize=8)
ax.scatter([b['k']], [b['aicc']], s=70, color='#0d366b', zorder=4)
ax.annotate(f"optimum k = {b['k']}", (b['k'], b['aicc']), xytext=(6, -12), textcoords='offset points', fontsize=8)
ax.set_ylim(bi['aicc']-1.5, g['aicc']+1.5)
ax.set_xlabel('Adaptive bandwidth (number of nearest provinces)'); ax.set_ylabel('AICc')
ax.grid(axis='y', color='#ececea', lw=0.6); ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout(); fig.savefig(f'{OUT}/Fig3_bandwidth.png', dpi=300); plt.show()

## 12. Validasi silang dengan `statsmodels`
Logit global dan OLS dari fungsi buatan sendiri harus sama dengan hasil `statsmodels`.

In [ ]:
try:
    import statsmodels.api as sm
    Xg = main['X']
    r = sm.Logit(y, Xg).fit(disp=0)
    print('Logit  statsmodels:', np.round(r.params, 4), '| buatan sendiri:', np.round(g['b'], 4))
    Xc = np.column_stack([np.ones(n), zs(m.Dry.values), zs(m.Heat.values)])
    r2 = sm.OLS(m.Yield_dev_trend.values*100, Xc).fit()
    print('OLS    statsmodels:', np.round(r2.params, 4), '| p Heat =', round(r2.pvalues[2], 4))
except ImportError:
    print('statsmodels tidak tersedia; lewati validasi.')

## 13. Unduh hasil

In [ ]:
import shutil
shutil.make_archive('hasil_gwlr', 'zip', OUT)
try:
    from google.colab import files
    files.download('hasil_gwlr.zip')
except ImportError:
    print('File hasil ada di folder:', os.path.abspath(OUT))